# RAG Evaluation Workshop

Learn how to evaluate RAG systems using various metrics and frameworks.

### What You'll Learn
- How to evaluate RAG systems using **DeepEval** (Ollama-based, no API key)
- Batch evaluation across multiple questions with progress tracking
- Understanding key metrics: Faithfulness, Answer Relevancy, Context Precision/Recall
- Evaluating your **actual RAG pipeline** (not just test data)
- Benchmarking latency and performance
- Saving results for analysis and regression testing

### Why Evaluation Matters
A RAG system is only as good as its outputs. Without evaluation, you can't:
- Detect when retrieval quality degrades
- Compare different RAG configurations
- Track improvements over time
- Ensure production quality


## Setup

### Why These Packages?
- **deepeval**: LLM evaluation framework that works with Ollama (no API key needed)
- **datasets**: HuggingFace data handling for structured evaluation data
- **tqdm**: Progress bars for batch evaluation


In [ ]:
!pip install deepeval datasets tqdm

## Prepare Evaluation Data

### Understanding Evaluation Data
Each question needs 4 components:
- **question**: What the user asks
- **answer**: The generated response (from your RAG or test data)
- **contexts**: Retrieved document chunks used to generate the answer
- **ground_truth**: The ideal/expected answer for comparison

> **Tip**: For real evaluations, use actual RAG outputs. For testing, you can use placeholder data.


In [1]:
from datasets import Dataset

# Sample evaluation data
eval_data = {
    "question": [
        "What is RAG?",
        "How does vector search work?",
        "What are the benefits of RAG?",
        "What is the difference between RAG and fine-tuning?",
        "When should you use RAG?"
    ],
    "answer": [
        "RAG stands for Retrieval-Augmented Generation...",
        "Vector search uses embeddings to find similar...",
        "RAG helps reduce hallucinations and provides...",
        "Fine-tuning adapts a model to specific tasks...",
        "You should use RAG when you need to query..."
    ],
    "contexts": [
        ["RAG is a framework...", "It combines retrieval..."],
        ["Vector embeddings...", "Similarity search..."],
        ["Benefits include...", "Reduced hallucinations..."],
        ["Fine-tuning is...", "RAG is..."],
        ["Use RAG for...", "External knowledge..."]
    ],
    "ground_truth": [
        "RAG is Retrieval-Augmented Generation, a technique...",
        "Vector search works by converting text to embeddings...",
        "RAG provides fresh knowledge, reduces hallucinations...",
        "Fine-tuning modifies model weights, RAG retrieves...",
        "Use RAG for question answering, knowledge bases..."
    ]
}

dataset = Dataset.from_dict(eval_data)
print(dataset)

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 5
})


## Evaluate with DeepEval


### Understanding the Metrics

| Metric | What it Measures | Why it Matters |
|--------|------------------|----------------|
| **Faithfulness** | Does the answer match the retrieved context? | Prevents hallucinations |
| **Answer Relevancy** | How relevant is the answer to the question? | User satisfaction |
| **Context Precision** | Are high-ranked chunks actually relevant? | Retrieval quality |
| **Context Recall** | Does retrieved context contain the answer? | Completeness |

All metrics return scores from 0-1, where 1.0 is perfect.


In [2]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric, ContextualPrecisionMetric, ContextualRecallMetric
from deepeval.models import OllamaModel
from typing import List, Dict
from tqdm import tqdm
import statistics

# Use Ollama for evaluation (no API key required)
ollama_model = OllamaModel(model="llama3.2")

# Initialize DeepEval metrics
faithfulness_metric = FaithfulnessMetric(model=ollama_model)
answer_relevancy_metric = AnswerRelevancyMetric(model=ollama_model)
context_precision_metric = ContextualPrecisionMetric(model=ollama_model)
context_recall_metric = ContextualRecallMetric(model=ollama_model)

print("✓ DeepEval metrics initialized with Ollama")
print("  - FaithfulnessMetric")
print("  - AnswerRelevancyMetric")
print("  - ContextualPrecisionMetric")
print("  - ContextualRecallMetric")


✓ DeepEval metrics initialized with Ollama
  - FaithfulnessMetric
  - AnswerRelevancyMetric
  - ContextualPrecisionMetric
  - ContextualRecallMetric


## Batch Evaluation Function


### Why Batch Evaluation?
Evaluating one question doesn't tell the whole story. Batch evaluation:
- Shows **consistency** across different questions
- Identifies **weak spots** in your RAG system
- Provides **statistical significance** (mean ± std)
- Enables **regression testing** over time

The `print_summary()` function calculates:
- **Average**: Overall performance
- **Std Dev**: How consistent the system is
- **Range**: Best and worst performing questions


In [3]:
def evaluate_batch(dataset, verbose: bool = True) -> List[Dict]:
    """Evaluate all questions in dataset with DeepEval metrics."""
    results = []
    
    for i in tqdm(range(len(dataset)), desc="Evaluating", disable=not verbose):
        test_case = LLMTestCase(
            input=dataset[i]["question"],
            actual_output=dataset[i]["answer"],
            retrieval_context=dataset[i]["contexts"],
            expected_output=dataset[i]["ground_truth"]
        )
        
        faithfulness_metric.measure(test_case)
        answer_relevancy_metric.measure(test_case)
        context_precision_metric.measure(test_case)
        context_recall_metric.measure(test_case)
        
        result = {
            "question": dataset[i]["question"],
            "faithfulness": faithfulness_metric.score,
            "answer_relevancy": answer_relevancy_metric.score,
            "context_precision": context_precision_metric.score,
            "context_recall": context_recall_metric.score
        }
        results.append(result)
        
        if verbose:
            print(f"\nQ{i+1}: {dataset[i]['question'][:50]}...")
            print(f"  Faithfulness: {faithfulness_metric.score:.2f}")
            print(f"  Answer Relevancy: {answer_relevancy_metric.score:.2f}")
            print(f"  Context Precision: {context_precision_metric.score:.2f}")
            print(f"  Context Recall: {context_recall_metric.score:.2f}")
    
    return results


def print_summary(results: List[Dict]) -> None:
    """Print summary statistics for evaluation results."""
    metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
    
    print("\n" + "="*60)
    print("EVALUATION SUMMARY")
    print("="*60)
    print(f"Total Questions: {len(results)}")
    print("-"*60)
    
    for metric in metrics:
        scores = [r[metric] for r in results]
        avg = statistics.mean(scores)
        std = statistics.stdev(scores) if len(scores) > 1 else 0
        print(f"{metric.replace('_', ' ').title()}: {avg:.2f} ± {std:.2f}")
    
    print("="*60)


In [4]:
# Run batch evaluation on all 5 questions
print("Running batch evaluation on all questions...\n")
evaluation_results = evaluate_batch(dataset, verbose=True)
print_summary(evaluation_results)


Running batch evaluation on all questions...



Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

Output()

Output()

Output()

...

Output()

Output()

Output()

Evaluating: 100%|██████████| 5/5 [00:53<00:00, 10.69s/it]


Q5: When should you use RAG?...
  Faithfulness: 0.67
  Answer Relevancy: 1.00
  Context Precision: 1.00
  Context Recall: 1.00

EVALUATION SUMMARY
Total Questions: 5
------------------------------------------------------------
Faithfulness: 0.13 ± 0.30
Answer Relevancy: 0.54 ± 0.46
Context Precision: 0.98 ± 0.04
Context Recall: 1.00 ± 0.00


### Interpreting Results

**Placeholder Data vs Real RAG**:
- This cell evaluates placeholder/test data (Cell 4)
- Low faithfulness = test data doesn't match contexts well
- This validates your test dataset quality

**To evaluate your REAL RAG pipeline**, see the section below.


## Custom LLM-as-Judge Evaluator

In [5]:
from langchain_ollama import ChatOllama

class RAGEvaluator:
    """Evaluate RAG using LLM as judge."""
    
    def __init__(self, llm=None):
        self.llm = llm or ChatOllama(model="llama3.2")
    
    def evaluate_faithfulness(self, question, answer, context):
        prompt = f"""Rate faithfulness 0-10:\n\nQuestion: {question}\n\nContext: {context}\n\nAnswer: {answer}\n\nOnly respond with a number:"""
        return float(self.llm.invoke(prompt).content.strip()) / 10
    
    def evaluate_answer_quality(self, question, answer):
        prompt = f"""Rate answer quality 0-10:\n\nQuestion: {question}\n\nAnswer: {answer}\n\nOnly respond with a number:"""
        return float(self.llm.invoke(prompt).content.strip()) / 10
    
    def full_evaluation(self, question, answer, contexts):
        context_str = "\n\n".join(contexts)
        return {
            "faithfulness": self.evaluate_faithfulness(question, answer, context_str),
            "answer_quality": self.evaluate_answer_quality(question, answer)
        }

evaluator = RAGEvaluator()

# Evaluate all questions
for i in range(len(eval_data["question"])):
    result = evaluator.full_evaluation(
        eval_data["question"][i],
        eval_data["answer"][i],
        eval_data["contexts"][i]
    )
    print(f"Q{i+1}: Faithfulness={result['faithfulness']:.2f}, Quality={result['answer_quality']:.2f}")

Q1: Faithfulness=0.80, Quality=0.60
Q2: Faithfulness=0.80, Quality=0.60
Q3: Faithfulness=0.80, Quality=0.60
Q4: Faithfulness=0.80, Quality=0.60
Q5: Faithfulness=0.60, Quality=0.60


## Retrieval Metrics

In [6]:
def precision_at_k(retrieved, relevant, k):
    retrieved_k = retrieved[:k]
    return len(set(retrieved_k) & set(relevant)) / k

def recall_at_k(retrieved, relevant, k):
    retrieved_k = retrieved[:k]
    return len(set(retrieved_k) & set(relevant)) / len(relevant)

# Example
retrieved = ["doc1", "doc2", "doc3", "doc4", "doc5"]
relevant = ["doc1", "doc3", "doc5"]

print(f"Precision@3: {precision_at_k(retrieved, relevant, 3):.2f}")
print(f"Recall@3: {recall_at_k(retrieved, relevant, 3):.2f}")
print(f"Precision@5: {precision_at_k(retrieved, relevant, 5):.2f}")
print(f"Recall@5: {recall_at_k(retrieved, relevant, 5):.2f}")

Precision@3: 0.67
Recall@3: 0.67
Precision@5: 0.60
Recall@5: 1.00


## Benchmarking Function

### Why Benchmark?
Latency matters for user experience:
- < 200ms: Excellent
- 200-500ms: Good
- 500-1000ms: Acceptable
- > 1000ms: Needs optimization

This benchmark creates a working RAG pipeline you can evaluate in the next section.


In [12]:
import time
from langchain_core.runnables import RunnableLambda

def benchmark_rag(rag_pipeline, test_queries):
    """Benchmark a RAG pipeline."""
    results = []
    
    for query in test_queries:
        start = time.time()
        response = rag_pipeline.invoke({"query": query})
        latency = (time.time() - start) * 1000
        
        # Handle different response types
        if isinstance(response, dict):
            answer = response.get("result", str(response))
        else:
            answer = str(response)
        
        results.append({
            "query": query,
            "latency_ms": latency,
            "answer": answer[:100]
        })
    
    avg_latency = sum(r["latency_ms"] for r in results) / len(results)
    
    return {
        "results": results,
        "avg_latency_ms": avg_latency,
        "total_queries": len(results)
    }

# ===== WORKING EXAMPLE =====

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Sample documents
documents = [
    "RAG stands for Retrieval-Augmented Generation. It combines a retrieval system with a generative AI model.",
    "RAG helps reduce hallucinations by grounding responses in retrieved context.",
    "Benefits of RAG include: fresh knowledge, reduced hallucinations, traceability, and cost-effective updates.",
    "Vector search works by converting text into embeddings and finding similar vectors.",
    "Embeddings are numerical representations of text that capture semantic meaning."
]

# Create embeddings and vector store
embeddings = OllamaEmbeddings(model="nomic-embed-text")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
docs = text_splitter.create_documents(documents)
vectorstore = InMemoryVectorStore.from_documents(docs, embedding=embeddings)
retriever = vectorstore.as_retriever()

# Create LLM
llm = ChatOllama(model="llama3.2")

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

def extract_question(inputs):
    """Extract question from dict or string."""
    if isinstance(inputs, dict):
        return inputs.get("query", inputs.get("question", ""))
    return inputs

def create_prompt(context, question):
    return f"""Based on the following context, answer the question.

Context: {context}

Question: {question}

Answer:"""

# Fixed RAG chain - extracts question, retrieves, then generates
simple_rag_chain = (
    RunnableLambda(extract_question)
    | retriever
    | format_docs
    | (lambda ctx: {"context": ctx, "question": extract_question({"query": ""})})
    | RunnableLambda(lambda x: llm.invoke(create_prompt(x["context"], x["question"])))
)

# Run benchmark
test_queries = [
    "What is RAG?",
    "How does it work?",
    "Benefits?"
]

print("Running benchmark...")
benchmark = benchmark_rag(simple_rag_chain, test_queries)
print(f"Average latency: {benchmark['avg_latency_ms']:.2f}ms")
print(f"Total queries: {benchmark['total_queries']}")
for r in benchmark['results']:
    print(f"  - {r['query']}: {r['latency_ms']:.2f}ms")

Running benchmark...
Average latency: 452.59ms
Total queries: 3
  - What is RAG?: 282.51ms
  - How does it work?: 152.76ms
  - Benefits?: 922.51ms


## Evaluate Actual RAG Pipeline Outputs


### Why Evaluate Real RAG Outputs?
This is the most important evaluation! 

- Uses your **actual RAG pipeline** (not placeholder data)
- Tests **retrieval + generation** together
- Shows real-world performance
- Faithfulness score here = your actual system quality


In [13]:
def evaluate_rag_pipeline(rag_chain, test_questions, verbose: bool = True) -> List[Dict]:
    """Evaluate actual RAG pipeline outputs with DeepEval metrics."""
    results = []
    
    for i in tqdm(range(len(test_questions)), desc="RAG Eval", disable=not verbose):
        question = test_questions[i]
        
        try:
            response = rag_chain.invoke(question)
            if isinstance(response, dict):
                answer = response.get("result", response.get("answer", str(response)))
            else:
                answer = str(response)
            
            docs = retriever.invoke(question)
            contexts = [doc.page_content for doc in docs]
            ground_truth = dataset[i]["ground_truth"] if i < len(dataset) else ""
            
            test_case = LLMTestCase(
                input=question,
                actual_output=answer,
                retrieval_context=contexts,
                expected_output=ground_truth
            )
            
            faithfulness_metric.measure(test_case)
            answer_relevancy_metric.measure(test_case)
            context_precision_metric.measure(test_case)
            context_recall_metric.measure(test_case)
            
            result = {
                "question": question,
                "answer": answer[:100],
                "faithfulness": faithfulness_metric.score,
                "answer_relevancy": answer_relevancy_metric.score,
                "context_precision": context_precision_metric.score,
                "context_recall": context_recall_metric.score
            }
            results.append(result)
            
            if verbose:
                print(f"\nQ{i+1}: {question}")
                print(f"  Faithfulness: { faithfulness_metric.score:.2f}")
        
        except Exception as e:
            print(f"Error: {e}")
    
    return results



In [14]:
test_questions_rag = ["What is RAG?", "How does it work?", "What are the benefits?"]

print("\n" + "="*60)
print("Evaluating ACTUAL RAG Pipeline Outputs")
print("="*60 + "\n")

rag_pipeline_results = evaluate_rag_pipeline(simple_rag_chain, test_questions_rag, verbose=True)

valid_results = [r for r in rag_pipeline_results if r.get("faithfulness") is not None]

if valid_results:
    print_summary(valid_results)



Evaluating ACTUAL RAG Pipeline Outputs



RAG Eval:   0%|          | 0/3 [00:00<?, ?it/s]

Output()

Output()

Output()

...

Output()

Output()

Output()

RAG Eval: 100%|██████████| 3/3 [00:49<00:00, 16.56s/it]


Q3: What are the benefits?
  Faithfulness: 1.00

EVALUATION SUMMARY
Total Questions: 3
------------------------------------------------------------
Faithfulness: 0.47 ± 0.48
Answer Relevancy: 0.92 ± 0.14
Context Precision: 1.00 ± 0.00
Context Recall: 1.00 ± 0.00


## Save Evaluation Results


### Why Save Results?
Saving evaluation results enables:
- **Regression testing**: Compare before/after changes
- **Historical tracking**: Monitor quality over time
- **Reporting**: Share results with stakeholders
- **Analysis**: Deep dive into specific failures

Files are timestamped for easy organization.


In [15]:
import json
from datetime import datetime

def save_results_to_json(results: List[Dict], filename: str = None) -> str:
    """Save evaluation results to JSON file."""
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"rag_evaluation_{timestamp}.json"
    
    export_data = {
        "timestamp": datetime.now().isoformat(),
        "total_questions": len(results),
        "results": results
    }
    
    with open(filename, 'w') as f:
        json.dump(export_data, f, indent=2)
    
    print(f"✓ Results saved to: {filename}")
    return filename


def results_to_csv(results: List[Dict], filename: str = None) -> str:
    """Convert evaluation results to CSV format."""
    import csv
    
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"rag_evaluation_{timestamp}.csv"
    
    columns = ["question", "answer", "faithfulness", "answer_relevancy", 
               "context_precision", "context_recall"]
    
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(results)
    
    print(f"✓ Results saved to: {filename}")
    return filename


# Save RAG pipeline results
if valid_results:
    json_file = save_results_to_json(valid_results)
    csv_file = results_to_csv(valid_results)
    print(f"\nSaved {len(valid_results)} evaluations to {json_file} and {csv_file}")


✓ Results saved to: rag_evaluation_20260310_134133.json
✓ Results saved to: rag_evaluation_20260310_134133.csv

Saved 3 evaluations to rag_evaluation_20260310_134133.json and rag_evaluation_20260310_134133.csv


## Next Steps

- Add more test questions (20-50 for statistical significance)
- Create evaluation dataset from real user queries
- Set up continuous evaluation
- Add visualizations (bar charts, radar charts)
- Compare different RAG configurations
- Integrate with Langfuse for production monitoring
